# Data Modeling

In [1]:
import os
import pandas as pd
from decimal import Decimal

# Load all the cleaned parquet files from the staging area
orders = pd.read_parquet("../data/staging/cleaned_orders.parquet", engine="pyarrow")
order_items = pd.read_parquet("../data/staging/cleaned_order_items.parquet", engine="pyarrow")
products = pd.read_parquet("../data/staging/cleaned_products.parquet", engine="pyarrow")
sellers = pd.read_parquet("../data/staging/cleaned_sellers.parquet", engine="pyarrow")
customers = pd.read_parquet("../data/staging/cleaned_customers.parquet", engine="pyarrow")
geolocation = pd.read_parquet("../data/staging/cleaned_geolocation.parquet", engine="pyarrow")

## 1. `dim_dates` table

In [2]:
# Collect all relevant dates
date_series = pd.concat([
    orders["purchase_timestamp"].dt.date,
    orders["approved_timestamp"].dt.date,
    orders["carrier_received_timestamp"].dt.date,
    orders["order_delivered_timestamp"].dt.date,
    orders["estimated_delivery_date"],
    order_items["ship_out_deadline"].dt.date
], ignore_index=True).dropna().unique()

dim_dates = pd.DataFrame({"date": pd.to_datetime(date_series)})

In [3]:
dim_dates["date_key"] = dim_dates["date"].dt.strftime("%Y%m%d").astype(int)
dim_dates["year"] = dim_dates["date"].dt.year
dim_dates["quarter"] = dim_dates["date"].dt.quarter
dim_dates["month"] = dim_dates["date"].dt.month
dim_dates["month_name"] = dim_dates["date"].dt.month_name()
dim_dates["week_of_year"] = dim_dates["date"].dt.isocalendar().week.astype(int)
dim_dates["day_of_month"] = dim_dates["date"].dt.day
dim_dates["day_of_week"] = dim_dates["date"].dt.weekday + 1 # convert from 0-6 to 1-7
dim_dates["day_name"] = dim_dates["date"].dt.day_name()
dim_dates["is_weekend"] = dim_dates["day_of_week"].isin([6, 7])

In [4]:
# Reorder columns
dim_dates = dim_dates[[
    "date_key",
    "date",
    "year",
    "quarter",
    "month",
    "month_name",
    "week_of_year",
    "day_of_month",
    "day_of_week",
    "day_name",
    "is_weekend"
]]

In [5]:
# Convert into appropriate data types to match ClickHouse schema
dim_dates["date_key"] = dim_dates["date_key"].astype("uint32")
# Convert date into date object
dim_dates["date"] = dim_dates["date"].dt.date
dim_dates["year"] = dim_dates["year"].astype("uint16")
dim_dates["quarter"] = dim_dates["quarter"].astype("uint8")
dim_dates["month"] = dim_dates["month"].astype("uint8")
dim_dates["month_name"] = dim_dates["month_name"].astype(str)
dim_dates["week_of_year"] = dim_dates["week_of_year"].astype("uint8")
dim_dates["day_of_month"] = dim_dates["day_of_month"].astype("uint8")
dim_dates["day_of_week"] = dim_dates["day_of_week"].astype("uint8")
dim_dates["day_name"] = dim_dates["day_name"].astype(str)
dim_dates["is_weekend"] = dim_dates["is_weekend"].astype(bool)

In [6]:
dim_dates.head()

,date_key,date,year,quarter,month,month_name,week_of_year,day_of_month,day_of_week,day_name,is_weekend
0,20171002,2017-10-02,2017,4,10,October,40,2,1,Monday,False
1,20180724,2018-07-24,2018,3,7,July,30,24,2,Tuesday,False
2,20180808,2018-08-08,2018,3,8,August,32,8,3,Wednesday,False
3,20171118,2017-11-18,2017,4,11,November,46,18,6,Saturday,True
4,20180214,2018-02-14,2018,1,2,February,7,14,3,Wednesday,False


In [7]:
# Check for missing values
print(dim_dates.isnull().sum())

# Check data types of each column
print(dim_dates.dtypes)

date_key        0
date            0
year            0
quarter         0
month           0
month_name      0
week_of_year    0
day_of_month    0
day_of_week     0
day_name        0
is_weekend      0
dtype: int64
date_key        uint32
date            object
year            uint16
quarter          uint8
month            uint8
month_name      object
week_of_year     uint8
day_of_month     uint8
day_of_week      uint8
day_name        object
is_weekend        bool
dtype: object


## 2. `dim_customers` table

In [8]:
# Check if the geolocation zip_code_prefix is unique
print(geolocation["zip_code_prefix"].is_unique)

# Check if the geolocation (zip_code_prefix, city, state) combination is unique
print(geolocation.duplicated(subset=["zip_code_prefix", "city", "state"]).sum() == 0)

# Check if customer_id is unique
print(customers["customer_id"].is_unique)

False
True
True


In [9]:
dim_customers = customers.copy()

# Add surrogate key
dim_customers["customer_key"] = dim_customers.index + 1

# Join with geolocation to get lat/lng
dim_customers = dim_customers.merge(
    geolocation,
    on=["zip_code_prefix", "city", "state"],
    how="left"
)

# Reorder columns
dim_customers = dim_customers[[
    "customer_key",
    "customer_id",
    "customer_unique_id",
    "zip_code_prefix",
    "city",
    "state",
    "lat",
    "lng"
]]

In [10]:
# Cast to appropriate data types to match ClickHouse schema
dim_customers["customer_key"] = dim_customers["customer_key"].astype("uint64")
dim_customers["customer_id"] = dim_customers["customer_id"].astype(str)
dim_customers["customer_unique_id"] = dim_customers["customer_unique_id"].astype(str)
dim_customers["zip_code_prefix"] = dim_customers["zip_code_prefix"].astype(str)
dim_customers["city"] = dim_customers["city"].astype(str)
dim_customers["state"] = dim_customers["state"].astype(str)
dim_customers["lat"] = dim_customers["lat"].astype("float64")
dim_customers["lng"] = dim_customers["lng"].astype("float64")

In [11]:
dim_customers.head()

,customer_key,customer_id,customer_unique_id,zip_code_prefix,city,state,lat,lng
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,-20.499273,-47.396658
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,-23.728396,-46.542250
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,-23.531309,-46.656690
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,-23.500670,-46.186348
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,-22.975708,-47.143140


In [12]:
# Check the shape before and after the merge
print("Customers shape:", customers.shape)
print("Dim Customers shape:", dim_customers.shape)

# Check for any missing values after the merge
print(dim_customers.isna().sum())

# Check the type of each column
print(dim_customers.dtypes)

Customers shape: (99441, 5)
Dim Customers shape: (99441, 8)
customer_key            0
customer_id             0
customer_unique_id      0
zip_code_prefix         0
city                    0
state                   0
lat                   302
lng                   302
dtype: int64
customer_key           uint64
customer_id            object
customer_unique_id     object
zip_code_prefix        object
city                   object
state                  object
lat                   float64
lng                   float64
dtype: object


## 3. `dim_sellers` table

In [13]:
# Check if the seller id is unique
print(sellers["seller_id"].is_unique)

True


In [14]:
dim_sellers = sellers.copy()

# Add surrogate key
dim_sellers["seller_key"] = dim_sellers.index + 1

# Join with geolocation to get lat/lng
dim_sellers = dim_sellers.merge(
    geolocation,
    on=["zip_code_prefix", "city", "state"],
    how="left"
)

# Reorder columns
dim_sellers = dim_sellers[[
    "seller_key",
    "seller_id",
    "zip_code_prefix",
    "city",
    "state",
    "lat",
    "lng"
]]

In [15]:
# Cast to appropriate data types to match ClickHouse schema
dim_sellers["seller_key"] = dim_sellers["seller_key"].astype("uint64")
dim_sellers["seller_id"] = dim_sellers["seller_id"].astype(str)
dim_sellers["zip_code_prefix"] = dim_sellers["zip_code_prefix"].astype(str)
dim_sellers["city"] = dim_sellers["city"].astype(str)
dim_sellers["state"] = dim_sellers["state"].astype(str)
dim_sellers["lat"] = dim_sellers["lat"].astype("float64")
dim_sellers["lng"] = dim_sellers["lng"].astype("float64")

In [16]:
dim_sellers.head()

,seller_key,seller_id,zip_code_prefix,city,state,lat,lng
0,1,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,-22.893317,-47.060596
1,2,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,-22.382914,-46.948763
2,3,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,-22.909446,-43.180240
3,4,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,-23.657118,-46.612730
4,5,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,-22.964546,-46.534214


In [17]:
# Check the shape before and after the merge
print("Sellers shape:", sellers.shape)
print("Dim Sellers shape:", dim_sellers.shape)

# Check for any missing values after the merge
print(dim_sellers.isna().sum())

# Check the type of each column
print(dim_sellers.dtypes)

Sellers shape: (3095, 4)
Dim Sellers shape: (3095, 7)
seller_key           0
seller_id            0
zip_code_prefix      0
city                 0
state                0
lat                135
lng                135
dtype: int64
seller_key          uint64
seller_id           object
zip_code_prefix     object
city                object
state               object
lat                float64
lng                float64
dtype: object


## 4. `dim_products` table

In [18]:
# Check if product_id is unique
print(products["product_id"].is_unique)

True


In [19]:
dim_products = products.copy()

# Add surrogate key
dim_products["product_key"] = dim_products.index + 1

# Reorder columns
dim_products = dim_products[[
    "product_key",
    "product_id",
    "category_name",
    "name_length",
    "description_length",
    "photos_quantity",
    "weight_g",
    "length_cm",
    "height_cm",
    "width_cm"
]]

In [20]:
# Cast to appropriate data types to match ClickHouse schema
dim_products["product_key"] = dim_products["product_key"].astype("uint64")
dim_products["product_id"] = dim_products["product_id"].astype(str)
dim_products["category_name"] = dim_products["category_name"].astype(str)
dim_products["name_length"] = dim_products["name_length"].astype("UInt32") # due to nulls, need use UInt32 instead of uint32
dim_products["description_length"] = dim_products["description_length"].astype("UInt32")
dim_products["photos_quantity"] = dim_products["photos_quantity"].astype("UInt16")
dim_products["weight_g"] = dim_products["weight_g"].astype("float64")
dim_products["length_cm"] = dim_products["length_cm"].astype("float64")
dim_products["height_cm"] = dim_products["height_cm"].astype("float64")
dim_products["width_cm"] = dim_products["width_cm"].astype("float64")

In [21]:
dim_products.head()

,product_key,product_id,category_name,name_length,description_length,photos_quantity,weight_g,length_cm,height_cm,width_cm
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumery,40,287,1,225.0,16.0,10.0,14.0
1,2,3aa071139cb16b67ca9e5dea641aaa2f,art,44,276,1,1000.0,30.0,18.0,20.0
2,3,96bd76ec8810374ed1b65e291975717f,sports_leisure,46,250,1,154.0,18.0,9.0,15.0
3,4,cef67bcfe19066a932b7673e239eb23d,baby,27,261,1,371.0,26.0,4.0,26.0
4,5,9dc1a7de274444849c219cff195d0b71,housewares,37,402,4,625.0,20.0,17.0,13.0


In [22]:
# Check the shape before and after
print("Products shape:", products.shape)
print("Dim Products shape:", dim_products.shape)

# Check for any missing values after the transformation
print(dim_products.isna().sum())

# Check the type of each column
print(dim_products.dtypes)

Products shape: (32951, 9)
Dim Products shape: (32951, 10)
product_key             0
product_id              0
category_name           0
name_length           610
description_length    610
photos_quantity       610
weight_g                2
length_cm               2
height_cm               2
width_cm                2
dtype: int64
product_key            uint64
product_id             object
category_name          object
name_length            UInt32
description_length     UInt32
photos_quantity        UInt16
weight_g              float64
length_cm             float64
height_cm             float64
width_cm              float64
dtype: object


## 5. `fact_order_items` table

In [23]:
# Check if order_id is unique in orders
print(orders["order_id"].is_unique)

# Check if (order_id, order_item_id) is unique in order_items
print(order_items.duplicated(subset=["order_id", "order_item_id"]).sum() == 0)

True
True


In [24]:
fact_order_items = order_items.copy()

# Merge order items with orders to get order-level details like delivery dates
fact_order_items = fact_order_items.merge(
    orders,
    on="order_id",
    how="left"
)

# Add customer surrogate key
fact_order_items = fact_order_items.merge(
    dim_customers[["customer_id", "customer_key"]],
    on="customer_id",
    how="left"
)

# Add seller surrogate key
fact_order_items = fact_order_items.merge(
    dim_sellers[["seller_id", "seller_key"]],
    on="seller_id",
    how="left"
)

# Add product surrogate key
fact_order_items = fact_order_items.merge(
    dim_products[["product_id", "product_key"]],
    on="product_id",
    how="left"
)

# Drop the natural keys in fact table
fact_order_items = fact_order_items.drop(
    columns=["customer_id", "product_id", "seller_id"]
)

In [25]:
# Create date keys
# Note: date keys are in YYYYMMDD format as integers
# We don't need to join with dim_dates since dim_dates is derived from these columns
# Other timestamps have NULL values so we won't force a dimension relationship and create keys for them
# "purchase_timestamp" is always present so we can create a date key for it
fact_order_items["purchase_date_key"] = fact_order_items["purchase_timestamp"].dt.strftime("%Y%m%d").astype(int)

In [26]:
# Add in a new measure
fact_order_items["item_total_value"] = fact_order_items["price"] + fact_order_items["freight_value"]

In [27]:
# Reorder columns, group like columns together 
# Low cardinality columns to the right
fact_order_items = fact_order_items[[
    # Identity / Grain (degenerate dimensions)
    "order_id",
    "order_item_id",
    
    # Foreign keys
    "customer_key",
    "seller_key",
    "product_key",
    "purchase_date_key",

    # Measures
    "price",
    "freight_value",
    "item_total_value",

    # Business lifecycle attributes
    "order_status",
    "purchase_timestamp",
    "approved_timestamp",
    "ship_out_deadline",
    "carrier_received_timestamp",
    "order_delivered_timestamp",
    "estimated_delivery_date",

    # Boolean data quality flags
    "missing_required_timestamps",
    "status_aware_ordering",
    "delivered_status_with_missing_timestamp",
    "delivered_timestamp_with_incorrect_status"
]]

In [28]:
def to_decimal_or_none(x):
    return Decimal(str(x)) if not pd.isna(x) else None

# Cast to appropriate data types to match ClickHouse schema
fact_order_items["order_id"] = fact_order_items["order_id"].astype(str)
fact_order_items["order_item_id"] = fact_order_items["order_item_id"].astype("uint32")
fact_order_items["customer_key"] = fact_order_items["customer_key"].astype("uint64")
fact_order_items["seller_key"] = fact_order_items["seller_key"].astype("uint64")
fact_order_items["product_key"] = fact_order_items["product_key"].astype("uint64")
fact_order_items["purchase_date_key"] = fact_order_items["purchase_date_key"].astype("uint32")
fact_order_items["price"] = fact_order_items["price"].apply(to_decimal_or_none)
fact_order_items["freight_value"] = fact_order_items["freight_value"].apply(to_decimal_or_none)
fact_order_items["item_total_value"] = fact_order_items["item_total_value"].apply(to_decimal_or_none)
fact_order_items["order_status"] = fact_order_items["order_status"].astype(str)

# Make the timestamp columns to seconds precision
fact_order_items["purchase_timestamp"] = fact_order_items["purchase_timestamp"].dt.floor('s')
fact_order_items["approved_timestamp"] = fact_order_items["approved_timestamp"].dt.floor('s')
fact_order_items["ship_out_deadline"] = fact_order_items["ship_out_deadline"].dt.floor('s')
fact_order_items["carrier_received_timestamp"] = fact_order_items["carrier_received_timestamp"].dt.floor('s')
fact_order_items["order_delivered_timestamp"] = fact_order_items["order_delivered_timestamp"].dt.floor('s')
# Convert estimated_delivery_date to date object
fact_order_items["estimated_delivery_date"] = pd.to_datetime(fact_order_items["estimated_delivery_date"], errors='coerce').dt.date

fact_order_items["missing_required_timestamps"] = fact_order_items["missing_required_timestamps"].astype(bool)
fact_order_items["status_aware_ordering"] = fact_order_items["status_aware_ordering"].astype(bool)
fact_order_items["delivered_status_with_missing_timestamp"] = fact_order_items["delivered_status_with_missing_timestamp"].astype(bool)
fact_order_items["delivered_timestamp_with_incorrect_status"] = fact_order_items["delivered_timestamp_with_incorrect_status"].astype(bool)

In [29]:
fact_order_items.head()

,order_id,order_item_id,customer_key,seller_key,product_key,purchase_date_key,price,freight_value,item_total_value,order_status,purchase_timestamp,approved_timestamp,ship_out_deadline,carrier_received_timestamp,order_delivered_timestamp,estimated_delivery_date,missing_required_timestamps,status_aware_ordering,delivered_status_with_missing_timestamp,delivered_timestamp_with_incorrect_status
0,00010242fe8c5a6d1ba2dd792cb16214,1,65558,514,25866,20170913,58.90,13.29,72.19,delivered,2017-09-13 11:59:02+00:00,2017-09-13 12:45:35+00:00,2017-09-19 12:45:35+00:00,2017-09-19 21:34:16+00:00,2017-09-21 02:43:48+00:00,2017-09-29,False,True,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,34266,472,27231,20170426,239.90,19.93,259.83,delivered,2017-04-26 13:53:06+00:00,2017-04-26 14:05:13+00:00,2017-05-03 14:05:13+00:00,2017-05-04 17:35:00+00:00,2017-05-12 19:04:24+00:00,2017-05-15,False,True,False,False
2,000229ec398224ef6ca0657da4fc703e,1,34956,1825,22625,20180114,199.00,17.87,216.87,delivered,2018-01-14 17:33:31+00:00,2018-01-14 17:48:30+00:00,2018-01-18 17:48:30+00:00,2018-01-16 15:36:48+00:00,2018-01-22 16:19:16+00:00,2018-02-05,False,True,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,51764,2024,15404,20180808,12.99,12.79,25.78,delivered,2018-08-08 13:00:35+00:00,2018-08-08 13:10:18+00:00,2018-08-15 13:10:18+00:00,2018-08-10 16:28:00+00:00,2018-08-14 16:32:39+00:00,2018-08-20,False,True,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,7603,1598,8863,20170204,199.90,18.14,218.04,delivered,2017-02-04 16:57:51+00:00,2017-02-04 17:10:13+00:00,2017-02-13 16:57:51+00:00,2017-02-16 12:46:09+00:00,2017-03-01 19:42:31+00:00,2017-03-17,False,True,False,False


In [30]:
# Check shape of fact table before and after
print("Order Items shape:", order_items.shape)
print("Fact Order Items shape:", fact_order_items.shape)

# Check for any missing values after the transformation
print(fact_order_items.isna().sum())

Order Items shape: (112650, 7)
Fact Order Items shape: (112650, 20)
order_id                                        0
order_item_id                                   0
customer_key                                    0
seller_key                                      0
product_key                                     0
purchase_date_key                               0
price                                           0
freight_value                                   0
item_total_value                                0
order_status                                    0
purchase_timestamp                              0
approved_timestamp                             15
ship_out_deadline                               0
carrier_received_timestamp                   1194
order_delivered_timestamp                    2454
estimated_delivery_date                         0
missing_required_timestamps                     0
status_aware_ordering                           0
delivered_status_with_missing_ti

In [31]:
# Check the type of each column
print(fact_order_items.dtypes)

order_id                                                  object
order_item_id                                             uint32
customer_key                                              uint64
seller_key                                                uint64
product_key                                               uint64
purchase_date_key                                         uint32
price                                                     object
freight_value                                             object
item_total_value                                          object
order_status                                              object
purchase_timestamp                           datetime64[ns, UTC]
approved_timestamp                           datetime64[ns, UTC]
ship_out_deadline                            datetime64[ns, UTC]
carrier_received_timestamp                   datetime64[ns, UTC]
order_delivered_timestamp                    datetime64[ns, UTC]
estimated_delivery_date  

## Uniqueness and Referential Integrity Check

In [32]:
print(dim_dates["date_key"].is_unique)
print(dim_customers["customer_key"].is_unique)
print(dim_sellers["seller_key"].is_unique)
print(dim_products["product_key"].is_unique)

# (order_id, order_item_id) is unique
print(fact_order_items.duplicated(subset=["order_id", "order_item_id"]).sum() == 0)

True
True
True
True
True


In [33]:
# Check that all foreign keys in fact table have matching entries in dimension tables
print(fact_order_items["customer_key"].isin(dim_customers["customer_key"]).all())
print(fact_order_items["seller_key"].isin(dim_sellers["seller_key"]).all())
print(fact_order_items["product_key"].isin(dim_products["product_key"]).all())
print(fact_order_items["purchase_date_key"].isin(dim_dates["date_key"]).all())

True
True
True
True


In [34]:
# Create data model directory if it doesn't exist
os.makedirs("../data/model", exist_ok=True)

# Save dimension tables
dim_dates.to_parquet("../data/model/dim_dates.parquet", engine="pyarrow", index=False)
dim_customers.to_parquet("../data/model/dim_customers.parquet", engine="pyarrow", index=False)
dim_sellers.to_parquet("../data/model/dim_sellers.parquet", engine="pyarrow", index=False)
dim_products.to_parquet("../data/model/dim_products.parquet", engine="pyarrow", index=False)
# Save fact table
fact_order_items.to_parquet("../data/model/fact_order_items.parquet", engine="pyarrow", index=False)